In [ ]:
# Locate the repository when Jupyter starts in a notebook subdirectory.
from pathlib import Path
import sys

_start = Path.cwd().resolve()
_repo = next((p for p in (_start, *_start.parents)
              if (p / "figure" / "paths.py").is_file()
              and (p / "run_cross_validation.py").is_file()), None)
if _repo is None:
    raise RuntimeError("Open this notebook inside the cloned sAge repository.")
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo))
from figure.paths import input_path, output_path, font_path


# figure-2-1-mouse-benchmark-MLP

Run mouse age-prediction MLP benchmarks for selected gene sets.

Run Jupyter from the repository root. Required external data and results are listed in `figure/INPUTS.md`. Set `SAGE_FIGURE_INPUT_ROOT` and `SAGE_FIGURE_OUTPUT_ROOT` when using other directories. See figure/VALIDATION.md for the execution checks and their limits.


1-benchmark

In [ ]:
import os

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import pandas as pd
import sys
import random
import re
import numpy as np
import hashlib

from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error # MAE, MSE
from scipy.stats import pearsonr
import h5py
from sklearn.preprocessing import StandardScaler
from joblib import Parallel, delayed
import warnings


# StandardScaler + MLPRegressor(hidden_layer_sizes=(64, 32), activation="tanh", alpha=0.01, max_iter=500, early_stopping=True)



benchmark_gene_dir = input_path('2-8.3-shanda/1-gene-benchmark')
gene_selection_thresholds = [5, 10, 15, 20, 25]
num_random_samples = 100 
N_JOBS = max(1, min(8, (os.cpu_count() or 2) - 1))

random.seed(42) 
np.random.seed(42) 
print("已设置全局初始随机种子为 42。")
print(f"并行进程数 N_JOBS = {N_JOBS}")

base_h5_data_dir = input_path('1-TMS-remove/2-restart')
header_file_path = input_path('header.txt')

model_configs = {
    "scimmuaging": {
        "base_output_dir": input_path("3-scimmuaging/output_5x"),
        "hdf5_base_dir": input_path("1-TMS-remove/2-restart"),
        "num_runs": 5,
        "file_pattern": "{tissue_name}_Fold{run_idx}_rankproduct_gene.txt",
        "dir_pattern": "Fold_{run_idx}"
    },
    "buckley": {
        "base_output_dir": input_path("6-buckley/1-5x"),
        "num_runs": 5,
        "file_pattern": "{tissue_name}_aging_related_genes_fold{run_idx}.csv",
        "dir_pattern": ""
    },
    "scale": {
        "base_output_dir": input_path("1-TMS-remove/kfold_results"),
        "num_runs": 5,
        "file_pattern": "initial_genes_fold{run_idx}_mapped.csv",
        "dir_pattern": "Fold_{run_idx}"
    },
    "maple": {
        "base_output_dir": input_path("2-8.3-shanda/1-feature/1-5x"),
        "num_runs": 5,
        "file_pattern": "feature.txt",
        "dir_pattern": "cv_folds/fold_{run_idx}/run",
        "header_path": input_path("2-8.3-shanda/header.txt")
    },
    "XGboost": {
        "base_output_dir": input_path("8-XGBoost_Paper/output"),
        "num_runs": 5,
        "file_pattern": "xist_genes.csv",
        "dir_pattern": "Fold_{run_idx}"
    },
    "iage": {
        "base_output_dir": input_path("1-deeplearn/2-iAge/1-output/2-gene-Mapped_Results"),
        "num_runs": 5,
        "file_pattern": "{tissue_name}_iAge_CV_Genes_Mapped.csv",
        "dir_pattern": ""
    }
}

age_label_to_months = {0: 1, 1: 3, 2: 18, 3: 21, 4: 24, 5: 30}
def map_labels_to_months(labels_array, mapping):
    return np.vectorize(mapping.get)(labels_array)


def evaluate_age_prediction(y_true, y_pred):
    if len(y_true) == 0 or len(y_pred) == 0:
        return np.nan, np.nan, np.nan
    
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    

    if len(np.unique(y_true)) < 2 or len(np.unique(y_pred)) < 2:
        pearson_r = np.nan
    else:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            pearson_r, _ = pearsonr(y_true, y_pred)
            
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return pearson_r, mae, rmse

def load_benchmark_genes(filepath):
    file_extension = os.path.splitext(filepath)[1].lower()
    df_benchmark = None
    gene_column_name = None

    try:
        if file_extension == '.csv':
            df_benchmark = pd.read_csv(filepath)
            if 'Symbol' in df_benchmark.columns: gene_column_name = 'Symbol'
            elif 'Gene' in df_benchmark.columns: gene_column_name = 'Gene'
            elif len(df_benchmark.columns) > 0: gene_column_name = df_benchmark.columns[0]
            else: raise ValueError(f"CSV文件 '{filepath}' 未找到合适的基因列。")
        elif file_extension == '.xlsx':
            df_benchmark = pd.read_excel(filepath)
            if 'Aging_map' in df_benchmark.columns: gene_column_name = 'Aging_map'
            elif 'Symbol' in df_benchmark.columns: gene_column_name = 'Symbol'
            elif 'Gene' in df_benchmark.columns: gene_column_name = 'Gene'
            elif len(df_benchmark.columns) > 0: gene_column_name = df_benchmark.columns[0]
            else: raise ValueError(f"Excel文件 '{filepath}' 未找到合适的基因列。")
        else: raise ValueError(f"不支持的文件类型")

        if gene_column_name is None: raise ValueError(f"未找到合适的基因列")

        return set(df_benchmark[gene_column_name].dropna().astype(str).str.strip().str.upper().tolist())
    except Exception as e:
        return None

def calculate_precision(predicted_genes_set, reference_set):
    if len(predicted_genes_set) == 0: return 0.0, 0
    matched = predicted_genes_set.intersection(reference_set)
    num_true_positives = len(matched)
    precision = num_true_positives / len(predicted_genes_set)
    return precision, num_true_positives

def get_tissues_for_model(model_name, config):
    tissues = set()
    base_dir = config['base_output_dir']
    if not os.path.exists(base_dir): return []

    if model_name == "scimmuaging":
        hdf5_base_dir = config.get('hdf5_base_dir')
        if not hdf5_base_dir or not os.path.exists(hdf5_base_dir): return []
        tissues = [d for d in os.listdir(hdf5_base_dir) if os.path.isdir(os.path.join(hdf5_base_dir, d))]
        if not tissues: return []
    elif model_name == "TDEseq":
        example_fold_dir = os.path.join(base_dir, config['dir_pattern'].format(run_idx=1))
        if not os.path.exists(example_fold_dir): return []
        pattern_suffix_template = config['file_pattern'].format(tissue_name="", run_idx="{run_idx}")
        pattern_suffix = pattern_suffix_template.format(run_idx=1).replace("__", "_")
        for filename in os.listdir(example_fold_dir):
            if filename.endswith(pattern_suffix):
                tissue_name = filename.replace(pattern_suffix, "")
                if tissue_name: tissues.add(tissue_name)
        if not tissues: return []
    elif model_name == "iage":
        for filename in os.listdir(base_dir):
            if filename.endswith("_iAge_CV_Genes_Mapped.csv"):
                tissue_name = filename.replace("_iAge_CV_Genes_Mapped.csv", "")
                tissues.add(tissue_name)
        if not tissues: return []
    elif model_name in ["buckley", "scale", "maple", "XGboost"]:
        tissues = [d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))]
        if not tissues: return []
    
    return sorted(list(tissues))

def process_model_output(model_name, config, benchmark_name, true_aging_genes, tissue_name, run_idx, threshold, num_random_samples, global_maple_header_genes=None):
    predicted_genes_file_path = None
    fold_dir_name_for_output = None
    ordered_all_predicted_genes_list = []

    if model_name in ["scimmuaging", "scale", "XGboost"]:
        fold_dir_name = config['dir_pattern'].format(run_idx=run_idx)
        current_output_dir_base = os.path.join(config['base_output_dir'], tissue_name, fold_dir_name)
        predicted_genes_file_name = config['file_pattern'].format(tissue_name=tissue_name, run_idx=run_idx)
        predicted_genes_file_path = os.path.join(current_output_dir_base, predicted_genes_file_name)
        fold_dir_name_for_output = f"Fold_{run_idx}"
    elif model_name == "buckley":
        current_output_dir_base = os.path.join(config['base_output_dir'], tissue_name)
        predicted_genes_file_name = config['file_pattern'].format(tissue_name=tissue_name, run_idx=run_idx)
        predicted_genes_file_path = os.path.join(current_output_dir_base, predicted_genes_file_name)
        fold_dir_name_for_output = f"fold{run_idx}"
    elif model_name == "maple":
        maple_run_base_dir = os.path.join(config['base_output_dir'], tissue_name, config['dir_pattern'].format(run_idx=run_idx))
        fold_dir_name_for_output = f"fold_{run_idx}"

        if not os.path.exists(maple_run_base_dir): return None, 2, []
        feature_dirs = []
        for d_name in os.listdir(maple_run_base_dir):
            match = re.match(r'feature(\d{6})', d_name)
            if match and os.path.isdir(os.path.join(maple_run_base_dir, d_name)):
                feature_dirs.append((int(match.group(1)), d_name))

        feature_dirs.sort()
        selected_feature_dir_name = None
        for num_genes_in_folder, d_name in feature_dirs:
            if num_genes_in_folder >= threshold:
                selected_feature_dir_name = d_name
                break

        if selected_feature_dir_name is None: return None, 1, []
        predicted_genes_file_path = os.path.join(maple_run_base_dir, selected_feature_dir_name, config['file_pattern'])

    elif model_name == "iage":
        current_output_dir_base = config['base_output_dir']
        predicted_genes_file_name = config['file_pattern'].format(tissue_name=tissue_name)
        predicted_genes_file_path = os.path.join(current_output_dir_base, predicted_genes_file_name)
        fold_dir_name_for_output = f"Fold_{run_idx}"
    else:
        return None, 2, []

    if not os.path.exists(predicted_genes_file_path):
        return None, 2, []
    else:
        try:
            file_extension = os.path.splitext(predicted_genes_file_path)[1].lower()

            if model_name == "maple":
                if global_maple_header_genes is None: return None, 2, []
                labels = []
                with open(predicted_genes_file_path, 'r', encoding='utf-8') as f:
                    for line in f:
                        stripped_line = line.strip()
                        if stripped_line and stripped_line.replace('.', '', 1).isdigit():
                            labels.append(float(stripped_line))

                if len(labels) != len(global_maple_header_genes): return None, 2, []
                for i, val in enumerate(labels):
                    if val == 1.000000:
                        ordered_all_predicted_genes_list.append(global_maple_header_genes[i])

            elif file_extension == '.csv':
                df_predicted = pd.read_csv(predicted_genes_file_path)
                if df_predicted.empty: return None, 2, []

                if model_name == "scale":
                    df_predicted.columns = df_predicted.columns.str.lower().str.replace('.', '_', regex=False)
                    up_col = next((col for col in df_predicted.columns if 'up' in col), None)
                    down_col = next((col for col in df_predicted.columns if 'down' in col), None)

                    if up_col is None or down_col is None: return None, 2, []

                    up_genes = df_predicted[up_col].dropna().astype(str).str.strip().str.upper().tolist()
                    down_genes = df_predicted[down_col].dropna().astype(str).str.strip().str.upper().tolist()
                    ordered_all_predicted_genes_list = up_genes + down_genes
                else:
                    gene_col = None
                    for col in df_predicted.columns:
                        col_lower = col.lower()
                        if col_lower in ['gene', 'symbol', 'gene_symbol', 'gene_name', 'features', 'feature']:
                            gene_col = col
                            break

                    if gene_col is None:
                        for col in df_predicted.columns:
                            if 'gene' in col.lower() or 'symbol' in col.lower():
                                gene_col = col
                                break

                    score_col = None
                    for col in df_predicted.columns:
                        col_lower = col.lower()
                        if any(keyword in col_lower for keyword in ['importance', 'score', 'weight']):
                            score_col = col
                            break

                    if gene_col:
                        if score_col:
                            df_predicted = df_predicted.drop_duplicates(subset=[gene_col])
                            df_predicted = df_predicted.sort_values(by=score_col, ascending=False)
                        ordered_all_predicted_genes_list = df_predicted[gene_col].dropna().astype(str).str.strip().str.upper().tolist()
                    else:
                        return None, 2, []

            elif file_extension == '.txt':
                with open(predicted_genes_file_path, 'r', encoding='utf-8') as f:
                    ordered_all_predicted_genes_list = [line.strip().upper() for line in f if line.strip()]
            else:
                return None, 2, []

            if not ordered_all_predicted_genes_list:
                return None, 2, []


            genes_for_prediction_pool = list(dict.fromkeys(ordered_all_predicted_genes_list))
            num_total_predicted_genes_in_file_unique = len(genes_for_prediction_pool)

            precision = 0.0
            num_true_positives = 0
            num_predicted_genes_after_selection = 0

            if num_total_predicted_genes_in_file_unique < threshold:
                return None, 1, []
            elif num_total_predicted_genes_in_file_unique == threshold:
                selected_predicted_genes_set = set(genes_for_prediction_pool)
                num_predicted_genes_after_selection = len(selected_predicted_genes_set)
                precision, num_true_positives = calculate_precision(selected_predicted_genes_set, true_aging_genes)
            else:
                precision_scores_for_averaging = []
                tp_counts_for_averaging = []


                seed_str = f"precision_{model_name}_{tissue_name}_{threshold}"
                p_seed = int(hashlib.md5(seed_str.encode('utf-8')).hexdigest(), 16) % (2**32)
                random.seed(p_seed)

                for _ in range(num_random_samples):
                    sampled_genes = set(random.sample(genes_for_prediction_pool, threshold))
                    current_precision, current_tp = calculate_precision(sampled_genes, true_aging_genes)
                    precision_scores_for_averaging.append(current_precision)
                    tp_counts_for_averaging.append(current_tp)

                precision = sum(precision_scores_for_averaging) / num_random_samples
                num_true_positives = round(sum(tp_counts_for_averaging) / num_random_samples)
                num_predicted_genes_after_selection = threshold

            result_data = {
                'Model': model_name,
                'Benchmark_Name': benchmark_name,
                'Tissue': tissue_name,
                'Run': fold_dir_name_for_output,
                'Threshold': threshold,
                'Predicted_Genes_Count': num_predicted_genes_after_selection,
                'True_Positives_Count': num_true_positives,
                'Precision': precision
            }
            return result_data, 0, genes_for_prediction_pool

        except Exception as e:
            return None, 2, []

if __name__ == "__main__":
    benchmark_files_map = {}
    if not os.path.exists(benchmark_gene_dir):
        print(f"错误：基准基因集目录 '{benchmark_gene_dir}' 不存在。")
        sys.exit(1)
    for f_name in os.listdir(benchmark_gene_dir):
        f_path = os.path.join(benchmark_gene_dir, f_name)
        if os.path.isfile(f_path) and (f_name.lower().endswith('.csv') or f_name.lower().endswith('.xlsx')):
            benchmark_files_map[f_name] = f_path
    if not benchmark_files_map:
        sys.exit(1)

    global_maple_header_genes = None
    if "maple" in model_configs:
        maple_header_path = model_configs["maple"]["header_path"]
        if os.path.exists(maple_header_path):
            with open(maple_header_path, 'r', encoding='utf-8') as f:
                global_maple_header_genes = [line.strip().upper() for line in f if line.strip()]

    real_gene_names = None
    try:
        with open(header_file_path, 'r') as f:
            real_gene_names = [line.strip().upper() for line in f if line.strip()]
    except Exception as e:
        sys.exit(1)

    all_tissue_names_from_h5_dir = sorted([d for d in os.listdir(base_h5_data_dir) if os.path.isdir(os.path.join(base_h5_data_dir, d))])

    loaded_benchmarks = {name: load_benchmark_genes(path) for name, path in benchmark_files_map.items()}
    loaded_benchmarks = {k: v for k, v in loaded_benchmarks.items() if v is not None}

    all_raw_results = []
    tissues_with_insufficient_genes = set()
    successfully_processed_combinations = set()
    prediction_pools_cache = {}

    print(f"\n--- [阶段 1] 开始解析模型基因并计算 Precision ---")

    for model_name, config in model_configs.items():
        tissues_for_model = get_tissues_for_model(model_name, config)
        common_tissues = sorted(list(set(tissues_for_model).intersection(all_tissue_names_from_h5_dir)))
        if not common_tissues: continue

        for tissue_name in common_tissues:
            target_run_idx_for_model = 4 if model_name == "maple" else 5
            if target_run_idx_for_model > config['num_runs']: continue

            for threshold in gene_selection_thresholds:
                pool_cached = False
                for benchmark_name, true_aging_genes in loaded_benchmarks.items():
                    if model_name == "maple":
                        res_data, status, pool = process_model_output(
                            model_name, config, benchmark_name, true_aging_genes,
                            tissue_name, target_run_idx_for_model, threshold, num_random_samples,
                            global_maple_header_genes=global_maple_header_genes)
                    else:
                        res_data, status, pool = process_model_output(
                            model_name, config, benchmark_name, true_aging_genes,
                            tissue_name, target_run_idx_for_model, threshold, num_random_samples)

                    if status == 1:
                        tissues_with_insufficient_genes.add((tissue_name, threshold))
                        continue 
                    elif status == 2:
                        continue

                    all_raw_results.append(res_data)
                    successfully_processed_combinations.add((model_name, tissue_name, threshold))

                    if not pool_cached:
                        prediction_pools_cache[(model_name, tissue_name, threshold)] = pool
                        pool_cached = True

    print(f"\n--- [阶段 2] 开始按组织并行加载 H5 数据并使用 mlp_tanh 进行年龄预测 ---")

    mlp_tanh_params = {
        "hidden_layer_sizes": (64, 32),
        "activation": "tanh",
        "alpha": 0.01,
        "max_iter": 500,
        "early_stopping": True,
    }


    real_gene_to_idx = {gene: idx for idx, gene in enumerate(real_gene_names)}

    def build_age_predictor(random_state):
        return MLPRegressor(
            **mlp_tanh_params,
            random_state=random_state,
        )

    def evaluate_one_gene_index_set(Data_train_scaled, Data_test_scaled, y_train, y_test, sampled_idx, seed):
        if not sampled_idx:
            return np.nan, np.nan, np.nan
        X_tr = Data_train_scaled[:, sampled_idx]
        X_te = Data_test_scaled[:, sampled_idx]
        if X_tr.shape[1] == 0:
            return np.nan, np.nan, np.nan
        model = build_age_predictor(seed)
        model.fit(X_tr, y_train)
        return evaluate_age_prediction(y_test, model.predict(X_te))

    def process_one_tissue_age_prediction(tissue_name, tissue_cache_keys):
        """Process one tissue per worker: read HDF5 once, then evaluate all models and thresholds."""
        if not tissue_cache_keys:
            return {}

        train_h5_file_path = os.path.join(base_h5_data_dir, tissue_name, 'train.h5')
        test_h5_file_path = os.path.join(base_h5_data_dir, tissue_name, 'test.h5')

        try:
            with h5py.File(train_h5_file_path, 'r') as f:
                Data_train = f['data'][:].astype(np.float32)
                Label_train = f['label'][:]
            y_train = map_labels_to_months(Label_train[:, 0].astype(np.int8), age_label_to_months)

            with h5py.File(test_h5_file_path, 'r') as f:
                Data_test = f['data'][:].astype(np.float32)
                Label_test = f['label'][:]
            y_test = map_labels_to_months(Label_test[:, 0].astype(np.int8), age_label_to_months)

            scaler = StandardScaler()
            Data_train_scaled = scaler.fit_transform(Data_train)
            Data_test_scaled = scaler.transform(Data_test)
        except Exception as e:
            print(f"  ! 跳过组织 {tissue_name}: H5 读取或标准化失败: {e}")
            return {}

        tissue_results = {}
        for cache_key in tissue_cache_keys:
            model_name, _, threshold = cache_key
            if (tissue_name, threshold) in tissues_with_insufficient_genes:
                continue

            genes_pool = prediction_pools_cache.get(cache_key, [])
            valid_gene_indices = [real_gene_to_idx[g] for g in genes_pool if g in real_gene_to_idx]

            pr_avg, mae_avg, rmse_avg = np.nan, np.nan, np.nan
            if valid_gene_indices:
                if len(valid_gene_indices) > threshold:
                    seed_str = f"regression_{model_name}_{tissue_name}_{threshold}"
                    current_seed = int(hashlib.md5(seed_str.encode('utf-8')).hexdigest(), 16) % (2**32)
                    rng = random.Random(current_seed)

                    pr_scores, ma_scores, rm_scores = [], [], []
                    sampled_index_sets = [rng.sample(valid_gene_indices, threshold) for _ in range(num_random_samples)]


                    for sampled_idx in sampled_index_sets:
                        pr, ma, rm = evaluate_one_gene_index_set(
                            Data_train_scaled, Data_test_scaled, y_train, y_test, sampled_idx, current_seed
                        )
                        if not np.isnan(pr):
                            pr_scores.append(pr)
                        if not np.isnan(ma):
                            ma_scores.append(ma)
                        if not np.isnan(rm):
                            rm_scores.append(rm)

                    if pr_scores:
                        pr_avg = float(np.mean(pr_scores))
                    if ma_scores:
                        mae_avg = float(np.mean(ma_scores))
                    if rm_scores:
                        rmse_avg = float(np.mean(rm_scores))
                else:
                    seed_str = f"mlp_tanh_{model_name}_{tissue_name}_{threshold}"
                    current_seed = int(hashlib.md5(seed_str.encode('utf-8')).hexdigest(), 16) % (2**32)
                    pr_avg, mae_avg, rmse_avg = evaluate_one_gene_index_set(
                        Data_train_scaled, Data_test_scaled, y_train, y_test, valid_gene_indices, current_seed
                    )

            tissue_results[cache_key] = {
                'Pearson_R': pr_avg,
                'MAE': mae_avg,
                'RMSE': rmse_avg,
            }

        print(f"  > 完成组织: {tissue_name}, 组合数: {len(tissue_results)}")
        return tissue_results

    tissue_to_cache_keys = {}
    for cache_key in prediction_pools_cache.keys():
        _, tissue_name, _ = cache_key
        tissue_to_cache_keys.setdefault(tissue_name, []).append(cache_key)

    tissue_tasks = [
        (tissue_name, cache_keys)
        for tissue_name, cache_keys in sorted(tissue_to_cache_keys.items())
        if cache_keys
    ]

    print(f"需要进行年龄预测的组织数: {len(tissue_tasks)}")
    tissue_result_list = Parallel(n_jobs=N_JOBS, backend="loky", verbose=10)(
        delayed(process_one_tissue_age_prediction)(tissue_name, cache_keys)
        for tissue_name, cache_keys in tissue_tasks
    )

    age_prediction_results = {}
    for tissue_result in tissue_result_list:
        age_prediction_results.update(tissue_result)

    for result in all_raw_results:
        key = (result['Model'], result['Tissue'], result['Threshold'])
        if key in age_prediction_results:
            result.update(age_prediction_results[key])
        else:
            result.update({'Pearson_R': np.nan, 'MAE': np.nan, 'RMSE': np.nan})

    num_total_models = len(model_configs)
    final_tissues_to_keep = set()

    model_counts_per_tissue_threshold = {}
    for model, tissue, threshold in successfully_processed_combinations:
        key = (tissue, threshold)
        model_counts_per_tissue_threshold.setdefault(key, set()).add(model)

    print(f"\n--- 筛选最终要保留的 (组织, 阈值) 对 ---")
    for (tissue, threshold), models_set in model_counts_per_tissue_threshold.items():
        if len(models_set) == num_total_models:
            if (tissue, threshold) not in tissues_with_insufficient_genes:
                final_tissues_to_keep.add((tissue, threshold))
        else:
            missing_models = set(model_configs.keys()) - models_set
            print(f"  排除 (组织: {tissue}, 阈值: {threshold})，未能处理模型: {', '.join(missing_models)}")

    filtered_results = [r for r in all_raw_results if (r['Tissue'], r['Threshold']) in final_tissues_to_keep]
    results_df = pd.DataFrame(filtered_results)

    print(f"\n--- 所有 Precision 和年龄预测结果 ---")
    print(results_df)

    if not results_df.empty:
        avg_results_summary = results_df.groupby(['Model', 'Benchmark_Name', 'Tissue', 'Threshold'])[
            ['Precision', 'Pearson_R', 'MAE', 'RMSE']
        ].mean().reset_index()
        print(avg_results_summary)
    else:
        avg_results_summary = pd.DataFrame()

    output_dir_for_results = input_path("1-TMS-remove/4-precision-5_to_25/1-5_to_25-pcc-add-model-mlp-tanh-parallel")
    os.makedirs(output_dir_for_results, exist_ok=True)

    output_full_results_file = os.path.join(output_dir_for_results, f"1-end-mlp_tanh_random_sample_parallel.csv")
    results_df.to_csv(output_full_results_file, index=False)
    
    avg_results_summary_file = os.path.join(output_dir_for_results, f"1-end-mlp_tanh_random_sample_summary_parallel.csv")
    avg_results_summary.to_csv(avg_results_summary_file, index=False)

    print("\n所有分析完成！数据已完全固化且可复现！")